# 02. Single-Experiment Workbook Walkthrough

## Purpose
This notebook is the canonical workbook-governed companion to notebook 01.
It demonstrates one single experiment executed through the workbook workflow with governance-first controls:
1. template generation/loading,
2. enabling exactly one executable row,
3. workbook validation/compile,
4. dry-run execution (default),
5. optional full execution,
6. write-back and provenance inspection.

## When to use this notebook
Use this notebook when you want workbook-governed execution, traceable planning metadata, and machine write-back.
For direct registry execution without workbook governance, use `01_single_experiment_registry_walkthrough.ipynb`.

## Prerequisites
- Repository checkout with environment prepared (for example: `uv sync --frozen --extra dev`)
- A valid dataset index CSV and data root for dry-run/full execution cells
- CLI access via `python -m Thesis_ML.cli.workbook` and `python -m Thesis_ML.cli.decision_support`

## Expected runtime
- Workbook setup + compile checks: short
- Dry-run: usually short
- Full execution: data-dependent and potentially much longer

## Expected outputs
Notebook-local files under:
`outputs/notebook_demo/02_single_experiment_workbook/`

## What this notebook does not cover
- Multi-experiment workbook campaigns
- Designed/factorial studies
- Scientific inference claims


## Configuration (edit this cell)

Set paths and run flags in this one cell. Defaults are portable and repo-relative.
Dry-run is enabled by default; full execution is opt-in.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

from Thesis_ML.demo.notebook_utils import (
    discover_project_root,
    execution_inputs_ready,
    path_status,
    resolve_user_path,
    setup_notebook_workspace,
)


PROJECT_ROOT = discover_project_root()
WORKSPACE = setup_notebook_workspace("02_single_experiment_workbook", project_root=PROJECT_ROOT)

# Optional overrides for dataset inputs.
INDEX_CSV = resolve_user_path(
    os.getenv("THESIS_ML_INDEX_CSV", ""),
    base=PROJECT_ROOT,
    default="Data/processed/dataset_index.csv",
)
DATA_ROOT = resolve_user_path(
    os.getenv("THESIS_ML_DATA_ROOT", ""),
    base=PROJECT_ROOT,
    default="Data",
)

# Notebook-local workspace.
DEMO_ROOT = WORKSPACE.notebook_root
WORKBOOK_ROOT = WORKSPACE.workbook_dir
OUTPUT_ROOT = WORKSPACE.output_root
WORKBOOK_OUTPUT_DIR = WORKSPACE.workbook_output_dir
CACHE_DIR = WORKSPACE.cache_dir

# Workbook/template paths.
SHIPPED_TEMPLATE = PROJECT_ROOT / "templates" / "thesis_experiment_program.xlsx"
WORKBOOK_PATH = WORKBOOK_ROOT / "single_experiment_campaign.xlsx"
PRISTINE_WORKBOOK_PATH = WORKBOOK_ROOT / "single_experiment_campaign_pristine.xlsx"

# Study/experiment selection and runtime flags.
WORKBOOK_EXPERIMENT_ID = os.getenv("THESIS_ML_WORKBOOK_EXPERIMENT_ID", "E16")
RUN_DRY_RUN = True
RUN_FULL_EXECUTION = False
REQUIRE_DRY_RUN_SUCCESS = False
RESET_NOTEBOOK_WORKBOOK = True

inputs_ready, input_issues = execution_inputs_ready(index_csv=INDEX_CSV, data_root=DATA_ROOT)
statuses = path_status(
    {
        "PROJECT_ROOT": PROJECT_ROOT,
        "SHIPPED_TEMPLATE": SHIPPED_TEMPLATE,
        "WORKBOOK_PATH": WORKBOOK_PATH,
        "PRISTINE_WORKBOOK_PATH": PRISTINE_WORKBOOK_PATH,
        "INDEX_CSV": INDEX_CSV,
        "DATA_ROOT": DATA_ROOT,
        "OUTPUT_ROOT": OUTPUT_ROOT,
        "WORKBOOK_OUTPUT_DIR": WORKBOOK_OUTPUT_DIR,
    }
)

config_preview = {
    "PROJECT_ROOT": statuses["PROJECT_ROOT"]["path"],
    "SHIPPED_TEMPLATE": statuses["SHIPPED_TEMPLATE"]["path"],
    "WORKBOOK_PATH": statuses["WORKBOOK_PATH"]["path"],
    "PRISTINE_WORKBOOK_PATH": statuses["PRISTINE_WORKBOOK_PATH"]["path"],
    "WORKBOOK_EXPERIMENT_ID": WORKBOOK_EXPERIMENT_ID,
    "INDEX_CSV": statuses["INDEX_CSV"]["path"],
    "INDEX_CSV_EXISTS": statuses["INDEX_CSV"]["exists"],
    "DATA_ROOT": statuses["DATA_ROOT"]["path"],
    "DATA_ROOT_EXISTS": statuses["DATA_ROOT"]["exists"],
    "EXECUTION_INPUTS_READY": inputs_ready,
    "EXECUTION_INPUT_ISSUES": input_issues,
    "OUTPUT_ROOT": statuses["OUTPUT_ROOT"]["path"],
    "WORKBOOK_OUTPUT_DIR": statuses["WORKBOOK_OUTPUT_DIR"]["path"],
    "RUN_DRY_RUN": RUN_DRY_RUN,
    "RUN_FULL_EXECUTION": RUN_FULL_EXECUTION,
    "RESET_NOTEBOOK_WORKBOOK": RESET_NOTEBOOK_WORKBOOK,
}
print(json.dumps(config_preview, indent=2))


In [ ]:
from Thesis_ML.demo.notebook_utils import (
    find_first_under,
    parse_cli_key_value_lines,
    preview_csv_rows,
    print_tree,
    read_json_safe,
    run_command,
)


def run(cmd: list[str], *, cwd: Path | None = None, check: bool = False):
    return run_command(cmd, cwd=cwd or PROJECT_ROOT, check=check)


def safe_read_json(path: Path):
    return read_json_safe(Path(path))


def preview_csv(path: Path, *, rows: int = 5) -> None:
    preview = preview_csv_rows(Path(path), n=rows)
    if not preview:
        return
    for row in preview:
        print(row)


def parse_campaign_paths(stdout_text: str) -> dict[str, str]:
    return parse_cli_key_value_lines(stdout_text)


## Workbook path role in this project

In this framework, the workbook is a governed planning interface, not just an input file.
The workflow is:
1. human-authored planning/governance fields,
2. workbook validation and compilation,
3. execution using the existing engine,
4. machine write-back (`Trial_Results`, `Summary_Outputs`, `Study_Review`, and related sheets).

Difference vs notebook 01:
- Notebook 01 runs directly from the registry.
- Notebook 02 demonstrates workbook-governed execution for one selected experiment row.


## Create notebook-local workbook template copy

The notebook creates a workbook in its own output area so shipped template assets are never edited in place.


In [ ]:
from Thesis_ML.demo.notebook_utils import copy_file_to_workspace

workbook_cli_cmd = [sys.executable, "-m", "Thesis_ML.cli.workbook", "--output", str(WORKBOOK_PATH)]

if RESET_NOTEBOOK_WORKBOOK or not WORKBOOK_PATH.exists():
    run(workbook_cli_cmd, cwd=PROJECT_ROOT, check=True)
else:
    print(f"Keeping existing workbook: {WORKBOOK_PATH}")

copy_file_to_workspace(
    source_path=WORKBOOK_PATH,
    destination_path=PRISTINE_WORKBOOK_PATH,
    overwrite=True,
)
print(f"Workbook ready: {WORKBOOK_PATH}")
print(f"Pristine snapshot for non-runnable check: {PRISTINE_WORKBOOK_PATH}")


## Inspect workbook structure

Keep the inspection concise: this notebook focuses on single-experiment execution via `Experiment_Definitions`.


In [ ]:
from Thesis_ML.demo.notebook_utils import summarize_workbook_sheet
from openpyxl import load_workbook

wb_preview = load_workbook(WORKBOOK_PATH, data_only=False)
print(f"Workbook sheets ({len(wb_preview.sheetnames)}):")
print(wb_preview.sheetnames)

relevant_sheet_specs = {
    "Experiment_Definitions": (1, 2),
    "Study_Review": (2, 3),
    "Trial_Results": (2, 3),
    "Summary_Outputs": (2, 3),
    "Generated_Design_Matrix": (2, 3),
}

for sheet_name, (header_row, data_start_row) in relevant_sheet_specs.items():
    sheet_summary = summarize_workbook_sheet(
        WORKBOOK_PATH,
        sheet_name=sheet_name,
        header_row=header_row,
        data_start_row=data_start_row,
        max_rows=1,
    )
    headers = sheet_summary["headers"]
    print(f"\n[{sheet_name}] header row {header_row} ({len(headers)} columns)")
    print(headers)


## Confirm template non-runnable state before enabling rows

The shipped workbook template is intentionally non-runnable until at least one executable row is enabled.
This cell verifies that expected guardrail behavior on the pristine copy.


In [ ]:
from Thesis_ML.orchestration.workbook_compiler import (
    NoEnabledExecutableRowsError,
    compile_workbook_file,
)

pristine_check_ok = False
try:
    compile_workbook_file(PRISTINE_WORKBOOK_PATH)
    raise AssertionError(
        "Pristine workbook compiled unexpectedly. Expected no enabled executable rows."
    )
except NoEnabledExecutableRowsError:
    pristine_check_ok = True
    print("Expected non-runnable template behavior confirmed.")

pristine_check_ok


## Enable exactly one executable row safely

This cell enables one row in `Experiment_Definitions` and disables all other executable rows.
It leaves all edits confined to the notebook-local workbook copy.


In [ ]:
from Thesis_ML.demo.notebook_utils import enable_single_workbook_experiment

selection = enable_single_workbook_experiment(
    workbook_path=WORKBOOK_PATH,
    experiment_id=WORKBOOK_EXPERIMENT_ID,
    default_values={
        "target": "coarse_affect",
        "cv": "within_subject_loso_session",
        "model": "ridge",
    },
)

WORKBOOK_EXPERIMENT_ID = str(selection["selected_experiment_id"])
if int(selection["enabled_rows"]) != 1:
    raise AssertionError(f"Expected exactly one enabled row, found {selection['enabled_rows']}")

selected_preview = {
    key: value
    for key, value in selection["row_payload"].items()
    if key
    in {
        "experiment_id",
        "enabled",
        "start_section",
        "end_section",
        "target",
        "cv",
        "model",
        "subject",
        "train_subject",
        "test_subject",
        "filter_task",
        "filter_modality",
        "reuse_policy",
        "search_space_id",
    }
}

print("Selected workbook row:")
print(json.dumps(selected_preview, indent=2))


## Validate and compile workbook

This step confirms that the edited workbook now compiles to a runnable single-experiment manifest.


In [ ]:
from Thesis_ML.workbook.validation import validate_workbook

validation_summary = validate_workbook(WORKBOOK_PATH)
print("Workbook validation summary (selected keys):")
for key in (
    "sheet_order_ok",
    "missing_sheets",
    "new_sheets_present",
    "required_named_lists_present",
    "experiment_definitions_columns_ok",
):
    print(f"- {key}: {validation_summary.get(key)}")

compiled_manifest = compile_workbook_file(WORKBOOK_PATH)
compiled_experiment_ids = [exp.experiment_id for exp in compiled_manifest.experiments]

print("\nCompiled workbook manifest:")
print(f"- experiments: {len(compiled_manifest.experiments)}")
print(f"- experiment_ids: {compiled_experiment_ids}")
print(f"- study_reviews: {len(compiled_manifest.study_reviews)}")

if len(compiled_manifest.experiments) != 1:
    raise AssertionError(
        f"Expected exactly one compiled experiment, got {len(compiled_manifest.experiments)}"
    )


## Dry-run execution (default path)

Dry-run is the primary demonstration path for this notebook.
It resolves the governed workbook plan and writes campaign outputs without full model execution.


In [ ]:
def build_workbook_campaign_command(*, dry_run: bool) -> list[str]:
    command = [
        sys.executable,
        "-m",
        "Thesis_ML.cli.decision_support",
        "--workbook",
        str(WORKBOOK_PATH),
        "--index-csv",
        str(INDEX_CSV),
        "--data-root",
        str(DATA_ROOT),
        "--cache-dir",
        str(CACHE_DIR),
        "--output-root",
        str(OUTPUT_ROOT),
        "--all",
        "--write-back-workbook",
        "--workbook-output-dir",
        str(WORKBOOK_OUTPUT_DIR),
    ]
    if dry_run:
        command.append("--dry-run")
    return command


DRY_RUN_RESULT: subprocess.CompletedProcess[str] | None = None
DRY_RUN_OK = False
DRY_RUN_META: dict[str, str] = {}

dry_run_inputs_ready, dry_run_issues = execution_inputs_ready(index_csv=INDEX_CSV, data_root=DATA_ROOT)
if RUN_DRY_RUN and dry_run_inputs_ready:
    DRY_RUN_RESULT = run(build_workbook_campaign_command(dry_run=True), cwd=PROJECT_ROOT, check=False)
    DRY_RUN_OK = DRY_RUN_RESULT.returncode == 0
    DRY_RUN_META = parse_campaign_paths(DRY_RUN_RESULT.stdout)
    print(f"Dry-run success: {DRY_RUN_OK}")
    if REQUIRE_DRY_RUN_SUCCESS and not DRY_RUN_OK:
        raise RuntimeError("Dry-run failed and REQUIRE_DRY_RUN_SUCCESS=True")
elif RUN_DRY_RUN:
    print("Dry-run skipped because required inputs are missing:")
    for issue in dry_run_issues:
        print(f"- {issue}")
else:
    print("Dry-run disabled by RUN_DRY_RUN=False")

DRY_RUN_META


## Optional full execution (guarded)

Full execution is intentionally opt-in and disabled by default.


In [ ]:
FULL_RUN_RESULT: subprocess.CompletedProcess[str] | None = None
FULL_RUN_OK = False
FULL_RUN_META: dict[str, str] = {}

full_run_inputs_ready, full_run_issues = execution_inputs_ready(index_csv=INDEX_CSV, data_root=DATA_ROOT)
if RUN_FULL_EXECUTION and full_run_inputs_ready:
    FULL_RUN_RESULT = run(build_workbook_campaign_command(dry_run=False), cwd=PROJECT_ROOT, check=False)
    FULL_RUN_OK = FULL_RUN_RESULT.returncode == 0
    FULL_RUN_META = parse_campaign_paths(FULL_RUN_RESULT.stdout)
    print(f"Full execution success: {FULL_RUN_OK}")
elif RUN_FULL_EXECUTION:
    print("Full execution requested but inputs are missing:")
    for issue in full_run_issues:
        print(f"- {issue}")
else:
    print("Full execution skipped (RUN_FULL_EXECUTION=False).")

FULL_RUN_META


## Inspect workbook write-back and campaign outputs

This section inspects machine-managed workbook outputs and campaign artifacts when available.


In [ ]:
campaign_meta = FULL_RUN_META if FULL_RUN_OK else DRY_RUN_META
campaign_root = Path(campaign_meta.get("campaign_root", "")) if campaign_meta.get("campaign_root") else None
manifest_path = (
    Path(campaign_meta["manifest"])
    if campaign_meta.get("manifest")
    else None
)

print("Notebook demo tree:")
print_tree(DEMO_ROOT, max_depth=4)

campaign_manifest = safe_read_json(manifest_path) if manifest_path else None
workbook_result_path = None
if isinstance(campaign_manifest, dict):
    workbook_output_path_text = campaign_manifest.get("exports", {}).get("workbook_output_path")
    if workbook_output_path_text:
        workbook_result_path = Path(workbook_output_path_text)

if workbook_result_path and workbook_result_path.exists():
    print(f"\nWorkbook write-back file: {workbook_result_path}")
    wb_out = load_workbook(workbook_result_path, data_only=True)
    for sheet_name, header_row in (
        ("Study_Review", 2),
        ("Trial_Results", 2),
        ("Summary_Outputs", 2),
        ("Generated_Design_Matrix", 2),
    ):
        ws = wb_out[sheet_name]
        print(f"\n[{sheet_name}] first populated rows:")
        shown = 0
        for row in ws.iter_rows(min_row=header_row, max_row=header_row + 6, values_only=True):
            if row and any(value not in (None, "") for value in row):
                print(row)
                shown += 1
        if shown == 0:
            print("(no populated rows found in preview window)")
else:
    print("\nNo workbook write-back file found yet (likely because execution was skipped or failed).")

print("\nCampaign root preview:")
if campaign_root:
    print_tree(campaign_root, max_depth=3)
else:
    print("No campaign root detected yet.")


## Inspect artifact provenance (SQLite registry)

Artifact lineage is central for auditability: it links run outputs to configs, upstream dependencies, and run status.


In [ ]:
from Thesis_ML.demo.notebook_utils import preview_artifact_registry

artifact_db_candidates = []
if campaign_root:
    artifact_db_candidates.extend(campaign_root.rglob("artifact_registry.sqlite3"))
artifact_db_candidates.extend(OUTPUT_ROOT.rglob("artifact_registry.sqlite3"))

artifact_db_path = artifact_db_candidates[0] if artifact_db_candidates else None
if artifact_db_path is None:
    print("Artifact registry database not found yet.")
else:
    preview = preview_artifact_registry(Path(artifact_db_path), limit=10)
    if not preview.get("exists"):
        print("Artifact registry database not found yet.")
    else:
        print(f"Artifact registry DB: {preview['path']}")
        print("Tables:", preview.get("tables", []))
        artifacts = preview.get("artifacts", [])
        if artifacts:
            print("\nRecent artifact records:")
            for row in artifacts:
                print(row)
        else:
            print("No 'artifacts' rows found in registry database preview.")


## Final summary


In [ ]:
important_names = [
    "campaign_manifest.json",
    "decision_support_summary.csv",
    "decision_recommendations.md",
    "result_aggregation.json",
    "study_review_summary.json",
    "summary_outputs_export.csv",
    "artifact_registry.sqlite3",
]

found_files = {
    name: [str(path) for path in OUTPUT_ROOT.rglob(name)]
    for name in important_names
}

summary = {
    "selected_workbook_experiment_id": WORKBOOK_EXPERIMENT_ID,
    "workbook_path": str(WORKBOOK_PATH),
    "pristine_template_guardrail_confirmed": pristine_check_ok,
    "dry_run_enabled": RUN_DRY_RUN,
    "dry_run_succeeded": DRY_RUN_OK,
    "full_execution_enabled": RUN_FULL_EXECUTION,
    "full_execution_succeeded": FULL_RUN_OK,
    "output_root": str(OUTPUT_ROOT),
    "important_files_found": sum(1 for paths in found_files.values() if paths),
    "important_file_paths": found_files,
}

print(json.dumps(summary, indent=2))


## What This Notebook Demonstrates

- Workbook-governed single-experiment setup using a notebook-local workbook copy
- Expected non-runnable template guardrail before enabling executable rows
- Safe enabling of exactly one executable row in `Experiment_Definitions`
- Workbook validation and compilation into a runnable manifest
- Dry-run campaign execution as the default demonstration path
- How write-back and artifact lineage can be inspected for auditability

## What This Notebook Does Not Establish Scientifically

- It does not establish causal validity, significance testing, or automatic scientific validity.
- It does not replace study design judgment, leakage review, or domain-specific protocol decisions.
- It demonstrates governed execution mechanics and provenance, not inferential proof.

Next notebook in this sequence: `03_*` notebooks for broader campaign or designed-study workflows.
